# WINGS v0.4.0 Feature Showcase

This notebook demonstrates features introduced in v0.4.0:
- **Composable Pipeline** system for multi-stage optimization
- **Tracy-Widom wavefunctions** as expressibility-hard targets
- **Multi-dimensional grids** (2D/3D Gaussians via `NDGrid`)
- **MPS initialization** for smarter starting parameters
- **Noise-aware optimization** with `NoiseConfig`
- **Natural gradient descent** with diagonal QFIM
- **Barren plateau detection**
- **Time evolution** via split-operator method

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

from wings import GaussianOptimizer, OptimizerConfig, TargetFunction
print(f"WINGS imported successfully")

## 1. Composable Pipeline

The `Pipeline` system chains optimization stages with automatic early stopping.

In [ ]:
from wings.pipeline import Pipeline, InitSearch, Adam, LBFGSB, Newton

# Build a custom pipeline: init -> Adam -> L-BFGS-B -> Newton polish
pipeline = Pipeline(
    target_fidelity=0.999,
    max_total_time=120,
    stages=[
        InitSearch(strategies=["smart", "random"]),
        Adam(max_steps=500, lr=0.01),
        LBFGSB(tolerances=[1e-10]),
        Newton(max_steps=20),
    ],
)

print(f"Pipeline stages: {len(pipeline.stages)}")
for i, stage in enumerate(pipeline.stages):
    print(f"  {i+1}. {stage.__class__.__name__}")

In [ ]:
# Use a preset pipeline
fast_pipe = Pipeline.fast(target_fidelity=0.99)
print("Fast pipeline stages:")
for s in fast_pipe.stages:
    print(f"  - {s.__class__.__name__}")

In [ ]:
# Run the pipeline on a 6-qubit Gaussian
config = OptimizerConfig(
    n_qubits=6,
    sigma=0.5,
    box_size=4.0,
    verbose=False,
    use_gpu=False,
    use_custatevec=False,
)
opt = GaussianOptimizer(config)

t0 = time.time()
result = opt.run_pipeline(pipeline)
elapsed = time.time() - t0

print(f"Fidelity:   {result['fidelity']:.10f}")
print(f"Infidelity: {1 - result['fidelity']:.4e}")
print(f"Time:       {elapsed:.1f}s")

## 2. Tracy-Widom Wavefunctions

Tracy-Widom distributions (beta = 1, 2, 4) serve as expressibility benchmarks.
They are harder for shallow circuits to represent than Gaussians.

In [ ]:
from wings.tracy_widom import (
    solve_painleve_ii,
    tracy_widom_distributions,
    tracy_widom_wavefunction,
    TW_BETA_1,
    TW_BETA_2,
    TW_BETA_4,
)

# Solve Painleve II and compute TW distributions
s_grid, q_vals, _ = solve_painleve_ii(s_max=5.0, s_min=-8.0, n_points=4096)
tw = tracy_widom_distributions(s_grid, q_vals)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PDFs
for beta_key, label in [("f1", r"$\beta=1$ (GOE)"), ("f2", r"$\beta=2$ (GUE)"), ("f4", r"$\beta=4$ (GSE)")]:
    axes[0].plot(s_grid, tw[beta_key], label=label, linewidth=2)
axes[0].set_xlabel("s")
axes[0].set_ylabel("f(s)")
axes[0].set_title("Tracy-Widom PDFs")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-6, 4)

# As quantum state targets
n_qubits = 6
x = np.linspace(-8, 5, 2**n_qubits)
for beta, label in [(TW_BETA_1, r"$\beta=1$"), (TW_BETA_2, r"$\beta=2$"), (TW_BETA_4, r"$\beta=4$")]:
    psi = tracy_widom_wavefunction(x, beta=beta)
    axes[1].plot(x, np.abs(psi)**2, label=label, linewidth=2)
axes[1].set_xlabel("x")
axes[1].set_ylabel(r"$|\psi(x)|^2$")
axes[1].set_title("Tracy-Widom as Quantum States")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Optimize toward a TW target
tw_config = OptimizerConfig(
    n_qubits=6,
    sigma=0.5,
    box_size=4.0,
    target_function=TargetFunction.TRACY_WIDOM_GOE,
    verbose=False,
    use_gpu=False,
    use_custatevec=False,
)
tw_opt = GaussianOptimizer(tw_config)

quick_pipe = Pipeline.fast(target_fidelity=0.95)
tw_result = tw_opt.run_pipeline(quick_pipe)
print(f"TW GOE fidelity: {tw_result['fidelity']:.6f}")

## 3. Multi-Dimensional Grids (NDGrid)

Prepare 2D and 3D Gaussian wavepackets using `NDGrid`.

In [ ]:
from wings.nd_grid import NDGrid, gaussian_nd

# 2D Gaussian on a 4+4 qubit grid (16x16 = 256 states)
grid_2d = NDGrid(n_qubits_per_dim=[4, 4], box_sizes=[4.0, 4.0])
psi_2d = gaussian_nd(grid_2d, sigmas=[0.5, 0.8], centers=[0.5, -0.3])

print(f"2D Grid: {grid_2d.n_dimensions}D, {grid_2d.total_qubits} qubits, {grid_2d.total_states} states")
print(f"Wavefunction shape: {psi_2d.shape}, norm: {np.linalg.norm(psi_2d):.10f}")

# Visualize
pos = grid_2d.positions()
X, Y = np.meshgrid(pos[0], pos[1], indexing="ij")
prob_2d = np.abs(psi_2d.reshape(16, 16))**2

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].pcolormesh(X, Y, prob_2d, cmap="viridis", shading="auto")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_title(r"2D Gaussian $|\psi(x,y)|^2$")
plt.colorbar(im, ax=axes[0])

# 3D grid (3+3+3 = 9 qubits)
grid_3d = NDGrid(n_qubits_per_dim=[3, 3, 3], box_sizes=[3.0, 3.0, 3.0])
psi_3d = gaussian_nd(grid_3d, sigmas=[0.5, 0.5, 0.5])
print(f"\n3D Grid: {grid_3d.n_dimensions}D, {grid_3d.total_qubits} qubits, {grid_3d.total_states} states")
print(f"Wavefunction shape: {psi_3d.shape}, norm: {np.linalg.norm(psi_3d):.10f}")

# Show marginal (integrate over y,z)
prob_3d = np.abs(psi_3d.reshape(8, 8, 8))**2
marginal_x = prob_3d.sum(axis=(1, 2))
axes[1].bar(range(8), marginal_x, color="steelblue")
axes[1].set_xlabel("x bin")
axes[1].set_ylabel("P(x)")
axes[1].set_title("3D Gaussian: x-marginal")

plt.tight_layout()
plt.show()

## 4. MPS Initialization

Matrix Product State decomposition provides smarter initial parameters
by matching single-qubit marginals of the target state.

In [ ]:
from wings.mps_init import mps_initial_params

config_8q = OptimizerConfig(
    n_qubits=8,
    sigma=0.5,
    box_size=4.0,
    verbose=False,
    use_gpu=False,
    use_custatevec=False,
)
opt_8q = GaussianOptimizer(config_8q)

# Compare random vs MPS initialization
np.random.seed(42)
random_params = np.random.randn(config_8q.n_params) * 0.1
mps_params = mps_initial_params(opt_8q.target, config_8q.n_qubits, config_8q.n_params)

fid_random = opt_8q.compute_fidelity(params=random_params)
fid_mps = opt_8q.compute_fidelity(params=mps_params)

print(f"Random init fidelity: {fid_random:.6f}")
print(f"MPS init fidelity:    {fid_mps:.6f}")
print(f"Improvement:          {fid_mps / max(fid_random, 1e-10):.1f}x")

## 5. Noise-Aware Optimization

`NoiseConfig` models hardware noise and provides noise-robust objectives.

In [ ]:
from wings.evaluators.noisy import NoiseConfig

# Typical superconducting hardware noise
noise = NoiseConfig(
    depolarizing_rate=0.001,
    amplitude_damping_rate=0.0005,
    gate_error_1q=0.001,
    gate_error_2q=0.01,
    readout_error=0.02,
)

print(f"Has noise: {noise.has_noise()}")
print(f"\nNoise-robust objective examples:")

# Compare: high ideal fidelity with/without noise gap
for f_ideal, f_noisy in [(0.99, 0.95), (0.99, 0.98), (0.95, 0.94)]:
    obj = noise.noise_robust_objective(f_ideal, f_noisy, robustness_weight=0.1)
    print(f"  F_ideal={f_ideal:.2f}, F_noisy={f_noisy:.2f} -> objective={obj:.4f}")

# Circuit depth penalty
for n_cx in [10, 50, 100]:
    penalty = noise.depth_penalty(n_cx)
    print(f"  {n_cx} CX gates -> depth penalty = {penalty:.4f}")

## 6. Natural Gradient & Barren Plateau Detection

In [ ]:
from wings.natural_gradient import compute_qfim_diagonal, compute_natural_gradient
from wings.barren_plateau import BarrenPlateauDetector

config_6q = OptimizerConfig(
    n_qubits=6,
    sigma=0.5,
    box_size=4.0,
    verbose=False,
    use_gpu=False,
    use_custatevec=False,
)
opt_6q = GaussianOptimizer(config_6q)
params_6q = opt_6q.get_initial_params("smart")

# Compute QFIM diagonal
qfim_diag = compute_qfim_diagonal(opt_6q, params_6q)
print(f"QFIM diagonal: shape={qfim_diag.shape}")
print(f"  min={qfim_diag.min():.6f}, max={qfim_diag.max():.6f}, mean={qfim_diag.mean():.6f}")

# Natural gradient
nat_grad = compute_natural_gradient(opt_6q, params_6q, regularization=0.001)
euc_grad = opt_6q.compute_gradient(params_6q)
print(f"\nEuclidean gradient norm: {np.linalg.norm(euc_grad):.6f}")
print(f"Natural gradient norm:   {np.linalg.norm(nat_grad):.6f}")

In [ ]:
# Barren plateau detection
detector = BarrenPlateauDetector(n_samples=20)

result = detector.analyze(opt_6q)

print(f"Barren plateau analysis (6 qubits):")
print(f"  Mean gradient variance: {result['mean_variance']:.6e}")
print(f"  Is barren:             {result['is_barren']}")
print(f"  Threshold used:        {result['threshold']:.6e}")

## 7. Time Evolution (Split-Operator)

Propagate a Gaussian wavepacket forward in time and use the result as a target.

In [ ]:
from wings.time_evolution import split_operator_step, make_grid

# Set up a 1D grid
n_points = 256
x, dx, k = make_grid(n_points, L=10.0)

# Initial Gaussian wavepacket with momentum
sigma0 = 0.5
k0 = 3.0  # initial momentum
psi0 = np.exp(-x**2 / (2 * sigma0**2)) * np.exp(1j * k0 * x)
psi0 /= np.linalg.norm(psi0)

# Free particle potential
V = np.zeros_like(x)

# Propagate
dt = 0.01
n_steps = 200
psi = psi0.copy()
snapshots = [np.abs(psi)**2]
times = [0.0]

for step in range(n_steps):
    psi = split_operator_step(psi, V, k, dt, mass=1.0)
    if (step + 1) % 50 == 0:
        snapshots.append(np.abs(psi)**2)
        times.append((step + 1) * dt)

# Verify norm conservation
final_norm = np.linalg.norm(psi)
print(f"Final norm: {final_norm:.10f} (should be ~1.0)")
print(f"Norm error: {abs(final_norm - 1.0):.2e}")

# Plot evolution
fig, ax = plt.subplots(figsize=(10, 4))
for prob, t in zip(snapshots, times):
    ax.plot(x, prob, label=f"t={t:.2f}", linewidth=1.5)
ax.set_xlabel("x")
ax.set_ylabel(r"$|\psi(x,t)|^2$")
ax.set_title("Free Particle Wavepacket Evolution")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(-5, 10)
plt.tight_layout()
plt.show()

## 8. Dashboard (Interactive Visualization)

The `OptimizationDashboard` tracks convergence in real time.

In [ ]:
from wings.dashboard import OptimizationDashboard

# Simulate an optimization trajectory
dash = OptimizationDashboard()

fidelities = 1.0 - np.exp(-np.linspace(0, 5, 50))  # fake convergence curve
for i, f in enumerate(fidelities):
    grad_norm = max(0.01, 1.0 - f) * np.random.uniform(0.8, 1.2)
    dash.record(step=i, fidelity=f, gradient_norm=grad_norm)

summary = dash.get_summary()
print("Dashboard summary:")
for key, val in summary.items():
    print(f"  {key}: {val}")

# Save HTML report (works without plotly)
dash.save_html("optimization_dashboard.html")
print("\nSaved: optimization_dashboard.html")

## Summary

All v0.4.0 features verified:
- Composable Pipeline with preset configurations
- Tracy-Widom targets from random matrix theory
- Multi-dimensional wavefunctions via NDGrid
- MPS-based smart initialization
- Noise modeling and robust objectives
- Natural gradient with QFIM
- Barren plateau detection
- Split-operator time evolution
- Interactive optimization dashboard